In [115]:
import random

import numpy as np


class Value:
    def __init__(self, data, _child=(), _op='', label=''):
        self.data = data
        self.prev = set(_child) # the neuron values feeding into this neuron. NN is like a DAG
        self._op = _op # the op that created this value
        self.label = label # Variable name of the node for better visualization
        self.grad = 0 # Derivative of the loss function w.r.t this value
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad * 1.0 # 1.0 coz this is an addition operation, so gradient disappears
            other.grad += out.grad * 1.0
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def __rmul__(self, other): # python will call rmul() & radd() by reversing the original self & other assignments.
        return self.__mul__(other)

    def __radd__(self, other):
        return self.__add__(other)

    def __pow__(self, power):
        assert isinstance(power, int) or isinstance(power, float)
        out = Value(self.data ** power, (self,), '**')
        def _backward():
            self.grad += out.grad * power * (self.data ** (power - 1))
        out._backward = _backward
        return out

    def __sub__(self, other):
        return self + (other * (-1))

    def __rsub__(self, other):
        return self + (other * (-1))

    def __truediv__(self, other):
        return self * (other**-1) # as a / b is equivalent to a * (1/b)

    def __repr__(self):
        return f"Value(data={self.data})"

    def tanh(self):
        out = Value(np.tanh(self.data), (self,), 'tanh')
        def _backward():
            self.grad += out.grad * (1 - np.tanh(self.data) ** 2)
        out._backward = _backward
        return out

    def exp(self):
        out = Value(np.exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.grad * np.exp(self.data) # or out.data, same value
        out._backward = _backward
        return out

    def backward(self):
        topo = [] 
        visited = set()
        def build_topo(v: Value):
            if v not in visited:
                visited.add(v)
                for child in v.prev:
                    build_topo(child)
                topo.append(v) # Output node enters the list at the end
        
        build_topo(self) # Topo sort starting at self, which is the output node
        self.grad = 1 # Gradient of output node w.r.t itself is 1
        for node in reversed(topo):
            node._backward() # Backpropagate after forward pass

# Let's create a DAG with Value class nodes
# Imagine these functions as one single neuron & one output, then try to evaluate

# out = tanh(x)
# L = 0.5 *(out-y)^2
# dL/dout = (out - y) * 1, which is out.grad
# dL/dself = dL/dout * dout/dself
# Use Topo sort to get the actual ordering of nodes, then do a forward pass & backward pass on it.
# Why accumulate gradients i.e. +=, assume gradient with two perspectives, first with self & other, then other & self with the expression "a+a", the value with self is 'self' in one case & 'other' in another case, hence we need to accumulate both cases
# a*2 = a.__mul__(2) & 2*a = 2.__mul__(a), hence we need to introduce a fallback function __rmul__().

In [138]:
from typing import List


class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x_in):
        act = sum((wi*xi for wi, xi in zip(self.w, x_in)), self.b)
        return act.tanh()

    def params(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x_in):
        return [neuron(x_in) for neuron in self.neurons]

    def params(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.params())
        return params

class MLP: # Defining a multi layer perceptron
    def __init__(self, nins, nouts: List):
        self.layer_sizes = [nins] + nouts
        self.layers = [Layer(self.layer_sizes[i], self.layer_sizes[i + 1]) for i in range(len(self.layer_sizes) - 1)]


    def __call__(self, x_in):
        for layer in self.layers:
            x_in =  layer(x_in)
        return x_in

    def params(self):
        params = []
        for layer in self.layers:
            params.extend(layer.params())
        return params

In [160]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
n(x) # Initialize a MLP

[Value(data=-0.9719129136295355)]

In [159]:
# Testing with an example dataset
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets for 4 data points in xs
 # Initial prediction

In [161]:
y_pred = [n(x) for x in xs]

In [162]:
loss = sum([(y_t - y_out[0])**2 for y_t, y_out in zip(ys, y_pred)]) # Need to minimize this loss
loss.backward()

Value(data=7.90984053663794)

In [166]:
count = 1
while count > 0:
    count = count - 1
    for p in n.params():
        p.data = p.data - 0.01 * p.grad
    y_pred = [n(x) for x in xs] # forward pass
    loss = sum([(y_t - y_out[0])**2 for y_t, y_out in zip(ys, y_pred)])
    for p in n.params():
        p.grad = 0.0 # Reset the grad value before next iteration, as grad calculation has to restart for new weights not accumulate, else grads would be very large and gradient descent would end up swinging around global minima
    loss.backward()

In [90]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v.prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)
  return dot